# IPPO (VMAS Navigation) - Colab Notebook

This notebook runs the IPPO implementation on the VMAS `navigation` scenario using a PettingZoo-style adapter.

1. In Colab, set runtime to **GPU**.
2. Run cells top-to-bottom.
3. Training logs and plots follow the same metric layout used in `gnn_mapp_vmas.ipynb`.


In [6]:
%pip -q install vmas matplotlib torch


In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import vmas


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class VMASAdapter:
    """
    Thin wrapper that exposes VMAS with a PettingZoo-parallel style API.
    """

    ACTION_DIM = 5  # no-op + left + right + up + down

    def __init__(self, n_agents, max_steps, device, seed=None):
        self.n_agents = n_agents
        self._device = torch.device(device)
        self._max_steps = max_steps
        self._step_count = 0
        self.obs_dim = None

        self.possible_agents = [f"agent_{i}" for i in range(n_agents)]
        self.agents = list(self.possible_agents)

        self._env = vmas.make_env(
            scenario="navigation",
            num_envs=1,
            n_agents=n_agents,
            device=str(self._device),
            continuous_actions=False,
            seed=seed,
        )

    def reset(self, seed=None):
        obs_list = self._env.reset(seed=seed) if seed is not None else self._env.reset()
        self._step_count = 0

        obs_dict = {
            aid: obs_list[i][0].detach().cpu().numpy()
            for i, aid in enumerate(self.possible_agents)
        }

        self.obs_dim = obs_list[0].shape[-1]
        return obs_dict, {}

    def step(self, actions_dict):
        action_list = [
            torch.tensor([actions_dict[aid]], device=self._device)
            for aid in self.possible_agents
        ]

        obs_list, rew_list, done_list, _ = self._env.step(action_list)
        self._step_count += 1
        truncated = self._step_count >= self._max_steps

        obs_dict, rew_dict, done_dict, trunc_dict = {}, {}, {}, {}
        for i, aid in enumerate(self.possible_agents):
            obs_dict[aid] = obs_list[i][0].detach().cpu().numpy()
            rew_dict[aid] = float(rew_list[i][0].item())
            agent_done = bool(done_list[i][0].item()) if done_list[i] is not None else False
            done_dict[aid] = agent_done or truncated
            trunc_dict[aid] = truncated

        return obs_dict, rew_dict, done_dict, trunc_dict, {}

    def close(self):
        close_fn = getattr(self._env, "close", None)
        if callable(close_fn):
            close_fn()


class ActorNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, actions_dim, device=None):
        super(ActorNetwork, self).__init__()
        self.device = torch.device(device) if device is not None else get_device()

        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, actions_dim)

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=0.01)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

    def get_action_and_log_probs(self, obs, action=None):
        logits = self.forward(obs)
        dist = Categorical(logits=logits)

        if action is None:
            action = dist.sample()
        else:
            if not torch.is_tensor(action):
                action = torch.as_tensor(action, dtype=torch.long, device=self.device)
            else:
                action = action.to(self.device, dtype=torch.long)

        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return action, log_prob, entropy

    def evaluate_actions(self, obs, actions):
        if not torch.is_tensor(actions):
            actions = torch.as_tensor(actions, dtype=torch.long, device=self.device)
        else:
            actions = actions.to(self.device, dtype=torch.long)

        logits = self.forward(obs)
        dist = Categorical(logits=logits)
        log_prob = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_prob, entropy, logits


class CriticNetwork(nn.Module):
    def __init__(self, obs_dims, hidden_dims, device=None):
        super(CriticNetwork, self).__init__()
        self.device = torch.device(device) if device is not None else get_device()

        self.fc1 = nn.Linear(obs_dims, hidden_dims)
        self.fc2 = nn.Linear(hidden_dims, hidden_dims)
        self.fc3 = nn.Linear(hidden_dims, 1)

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x


class RolloutBuffer:
    def __init__(self, buffer_size, obs_dim, action_dim, gamma, gae_lambda, device):
        self.buffer_size = buffer_size
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.device = device

        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []

    def add_rollout(self, obs, action, reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)

        self.rewards.append(reward.item() if torch.is_tensor(reward) else reward)
        self.dones.append(done.item() if torch.is_tensor(done) else done)
        self.log_probs.append(log_prob.item() if torch.is_tensor(log_prob) else log_prob)
        self.values.append(value.item() if torch.is_tensor(value) else value)

    def compute_returns_and_advantages(self, last_value):
        if len(self.rewards) == 0:
            raise RuntimeError("RolloutBuffer is empty. Collect rollouts before update().")

        rewards = torch.as_tensor(self.rewards, dtype=torch.float32, device=self.device)
        values = torch.as_tensor(self.values, dtype=torch.float32, device=self.device)
        dones = torch.as_tensor(self.dones, dtype=torch.float32, device=self.device)

        T = rewards.shape[0]
        advantages = torch.zeros(T, dtype=torch.float32, device=self.device)
        last_gae = 0.0

        for t in reversed(range(T)):
            if t == T - 1:
                next_value = (
                    torch.tensor(last_value, dtype=torch.float32, device=self.device)
                    if not torch.is_tensor(last_value)
                    else last_value.to(self.device, dtype=torch.float32)
                )
            else:
                next_value = values[t + 1]

            delta = rewards[t] + self.gamma * (1 - dones[t]) * next_value - values[t]
            advantages[t] = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            last_gae = advantages[t]

        self.advantages = advantages
        self.returns = advantages + values

    def get(self):
        obs = torch.as_tensor(np.asarray(self.obs), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(np.asarray(self.actions), dtype=torch.long, device=self.device)
        log_probs = torch.as_tensor(self.log_probs, dtype=torch.float32, device=self.device)

        adv = self.advantages
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        return obs, actions, log_probs, adv, self.returns

    def clear(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []


class PPOAgent:
    def __init__(
        self,
        obs_dim,
        hidden_dim,
        action_dim,
        lr=3e-4,
        buffer_size=2048,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        value_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5,
    ):
        self.device = get_device()

        self.actor = ActorNetwork(obs_dim, hidden_dim, action_dim, device=self.device).to(self.device)
        self.critic = CriticNetwork(obs_dim, hidden_dim, device=self.device).to(self.device)
        self.buffer = RolloutBuffer(buffer_size, obs_dim, action_dim, gamma, gae_lambda, self.device)

        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr=lr)

        self.clip_eps = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.gamma = gamma

    def select_action(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, _ = self.actor.get_action_and_log_probs(obs)
            value = self.critic.forward(obs)

        return actions.item(), log_probs.item(), value.squeeze().item()

    def update(self, last_obs, num_epochs=30):
        if len(self.buffer.rewards) == 0:
            return {
                "policy_loss": 0.0,
                "value_loss": 0.0,
                "entropy": 0.0,
                "mean_bellman_error": 0.0,
            }

        with torch.no_grad():
            last_obs_tensor = torch.as_tensor(last_obs, dtype=torch.float32, device=self.device).unsqueeze(0)
            last_value = self.critic(last_obs_tensor).squeeze()

        self.buffer.compute_returns_and_advantages(last_value)
        obs, actions, old_log_probs, advantages, returns = self.buffer.get()

        total_policy_loss = 0.0
        total_value_loss = 0.0
        total_entropy = 0.0

        for _ in range(num_epochs):
            new_log_probs, entropy, _ = self.actor.evaluate_actions(obs, actions)

            ratio = torch.exp(new_log_probs - old_log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            values = self.critic(obs).squeeze(-1)
            values_loss = F.mse_loss(values, returns)

            entropy_loss = -entropy.mean()
            loss = policy_loss + self.value_coef * values_loss + self.entropy_coef * entropy_loss

            self.actor_optim.zero_grad()
            self.critic_optim.zero_grad()
            loss.backward()

            nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
            nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)

            self.actor_optim.step()
            self.critic_optim.step()

            total_policy_loss += policy_loss.item()
            total_entropy += entropy.mean().item()
            total_value_loss += values_loss.item()

        with torch.no_grad():
            rewards_tensor = torch.as_tensor(self.buffer.rewards, dtype=torch.float32, device=self.device)
            values_tensor = torch.as_tensor(self.buffer.values, dtype=torch.float32, device=self.device)
            dones_tensor = torch.as_tensor(self.buffer.dones, dtype=torch.float32, device=self.device)

            next_values = torch.zeros_like(values_tensor)
            if len(values_tensor) > 1:
                next_values[:-1] = values_tensor[1:]
            next_values[-1] = last_value

            td_errors = rewards_tensor + self.gamma * (1 - dones_tensor) * next_values - values_tensor
            mean_bellman_error = td_errors.abs().mean().item()

        self.buffer.clear()

        return {
            "policy_loss": total_policy_loss / num_epochs,
            "value_loss": total_value_loss / num_epochs,
            "entropy": total_entropy / num_epochs,
            "mean_bellman_error": mean_bellman_error,
        }


class IPPOTrainer:
    def __init__(self, env, num_agents, obs_dim, hidden_dim, action_dim):
        self.env = env
        self.num_agents = num_agents
        self.agent_ids = sorted(env.possible_agents)
        self.agents = {
            agent_id: PPOAgent(obs_dim, hidden_dim, action_dim)
            for agent_id in self.agent_ids
        }

        self.metrics_history = {
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "mean_bellman_error": [],
            "mean_episode_return": [],
            "mean_episode_rewards": [],
        }
        self._running_episode_return = 0.0

    def _safe_mean(self, values):
        return float(sum(values) / len(values)) if values else 0.0

    def collect_rollouts(self, num_steps, obs):
        if obs is None:
            obs, _ = self.env.reset()

        step_mean_rewards = []
        completed_episode_returns = []

        for _ in range(num_steps):
            actions = {}
            values = {}
            log_probs = {}

            for a_id in self.agent_ids:
                action, log_prob, value = self.agents[a_id].select_action(obs[a_id])
                actions[a_id] = action
                values[a_id] = value
                log_probs[a_id] = log_prob

            next_obs, rewards, dones, truncs, _ = self.env.step(actions)

            for a_id in self.agent_ids:
                self.agents[a_id].buffer.add_rollout(
                    obs[a_id],
                    actions[a_id],
                    rewards[a_id],
                    dones[a_id],
                    log_probs[a_id],
                    values[a_id],
                )

            mean_step_reward = sum(rewards.values()) / max(len(rewards), 1)
            step_mean_rewards.append(mean_step_reward)
            self._running_episode_return += mean_step_reward

            if all(dones.values()) or all(truncs.values()):
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0
                obs, _ = self.env.reset()
            else:
                obs = next_obs

        rollout_metrics = {
            "mean_episode_return": self._safe_mean(completed_episode_returns)
            if completed_episode_returns
            else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
        }
        return obs, rollout_metrics

    def train(self, total_timesteps, rollout_length, initial_obs=None, log_every=10):
        steps_done = 0
        iteration = 0
        obs = initial_obs

        while steps_done < total_timesteps:
            rollout_steps = min(rollout_length, total_timesteps - steps_done)
            last_obs, rollout_metrics = self.collect_rollouts(rollout_steps, obs)

            all_agent_metrics = []
            for a_id in self.agent_ids:
                metrics = self.agents[a_id].update(last_obs[a_id])
                all_agent_metrics.append(metrics)

            avg_metrics = {
                key: sum(m[key] for m in all_agent_metrics) / len(all_agent_metrics)
                for key in all_agent_metrics[0].keys()
            }

            for k in ["policy_loss", "value_loss", "entropy", "mean_bellman_error"]:
                self.metrics_history[k].append(avg_metrics[k])
            self.metrics_history["mean_episode_return"].append(rollout_metrics["mean_episode_return"])
            self.metrics_history["mean_episode_rewards"].append(rollout_metrics["mean_episode_rewards"])

            steps_done += rollout_steps
            iteration += 1

            should_log = (
                iteration == 1
                or iteration % log_every == 0
                or steps_done == total_timesteps
            )
            if should_log:
                print(
                    f"Iter {iteration:4d} | "
                    f"steps={steps_done:>8}/{total_timesteps} | "
                    f"pi_loss={avg_metrics['policy_loss']:.4f} | "
                    f"v_loss={avg_metrics['value_loss']:.4f} | "
                    f"ent={avg_metrics['entropy']:.4f} | "
                    f"bellman={avg_metrics['mean_bellman_error']:.4f} | "
                    f"ep_ret={rollout_metrics['mean_episode_return']:.4f} | "
                    f"ep_rew={rollout_metrics['mean_episode_rewards']:.4f}"
                )

            obs = last_obs

    def plot_metrics(self, save_path="ippo_vmas_training_metrics.png", title="Training Metrics"):
        metrics_to_plot = [
            "policy_loss",
            "value_loss",
            "entropy",
            "mean_bellman_error",
            "mean_episode_return",
            "mean_episode_rewards",
        ]

        fig, axes = plt.subplots(3, 2, figsize=(14, 12))
        for ax, name in zip(axes.flatten(), metrics_to_plot):
            vals = self.metrics_history.get(name, [])
            ax.plot(range(1, len(vals) + 1), vals, linewidth=1.8)
            ax.set_title(name)
            ax.set_xlabel("Iteration")
            ax.set_ylabel(name)
            ax.grid(True, alpha=0.3)

        fig.suptitle(title)
        plt.tight_layout()
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Metrics plot saved to {save_path}")


print(f"Using device: {get_device()}")


Using device: cpu


In [8]:
# ============================================================
# Hyperparameters
# ============================================================
NUM_AGENTS = 10
MAX_CYCLES = 100
HIDDEN_DIM = 64
ACTION_DIM = VMASAdapter.ACTION_DIM

TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048
SEED = 42
LOG_EVERY = 10

OUTPUT_DIR = "outputs/ippo_vmas_navigation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = get_device()
print(f"Device: {device}")

# Infer obs_dim from VMAS env (matches gnn_mapp_vmas setup)
probe_env = VMASAdapter(n_agents=NUM_AGENTS, max_steps=MAX_CYCLES, device=device, seed=SEED)
probe_obs, _ = probe_env.reset(seed=SEED)
obs_dim = probe_env.obs_dim
probe_env.close()

print(f"obs_dim (inferred from env): {obs_dim}")
print(f"sample obs shape: {probe_obs['agent_0'].shape}")

train_env = VMASAdapter(n_agents=NUM_AGENTS, max_steps=MAX_CYCLES, device=device, seed=SEED)
initial_obs, _ = train_env.reset(seed=SEED)

trainer = IPPOTrainer(
    env=train_env,
    num_agents=NUM_AGENTS,
    obs_dim=obs_dim,
    hidden_dim=HIDDEN_DIM,
    action_dim=ACTION_DIM,
)

trainer.train(
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_length=ROLLOUT_LENGTH,
    initial_obs=initial_obs,
    log_every=LOG_EVERY,
)

plot_path = os.path.join(OUTPUT_DIR, "ippo_vmas_navigation_metrics.png")
trainer.plot_metrics(save_path=plot_path, title="IPPO Training Metrics (VMAS Navigation)")
print(f"Saved metrics plot: {plot_path}")


Observation shape: (30,)
Using Device: cpu


KeyboardInterrupt: 

In [ ]:
def evaluate_policy(trainer, num_agents, max_cycles, episodes=5, seed=123):
    eval_env = VMASAdapter(
        n_agents=num_agents,
        max_steps=max_cycles,
        device=get_device(),
        seed=seed,
    )

    obs, _ = eval_env.reset(seed=seed)
    completed = 0
    running_return = 0.0

    episode_returns = []
    step_rewards = []

    while completed < episodes:
        actions = {}
        for a_id in trainer.agent_ids:
            action, _, _ = trainer.agents[a_id].select_action(obs[a_id])
            actions[a_id] = action

        obs, rewards, dones, truncs, _ = eval_env.step(actions)
        mean_step_reward = sum(rewards.values()) / max(len(rewards), 1)
        step_rewards.append(mean_step_reward)
        running_return += mean_step_reward

        if all(dones.values()) or all(truncs.values()):
            episode_returns.append(running_return)
            running_return = 0.0
            completed += 1

            if completed < episodes:
                obs, _ = eval_env.reset(seed=seed + completed)

    eval_env.close()

    return {
        "episodes": episodes,
        "mean_episode_return": float(np.mean(episode_returns)),
        "std_episode_return": float(np.std(episode_returns)),
        "mean_episode_rewards": float(np.mean(step_rewards)),
    }


EVAL_EPISODES = 5
eval_metrics = evaluate_policy(
    trainer,
    num_agents=NUM_AGENTS,
    max_cycles=MAX_CYCLES,
    episodes=EVAL_EPISODES,
    seed=SEED + 100,
)

print("Evaluation metrics (VMAS Navigation):")
for k, v in eval_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

trainer.env.close()


Eval returns: {'agent_0': -28663.95449875764, 'agent_1': -29318.454498757685, 'agent_2': -28582.454498757663, 'agent_3': -29562.954498757677, 'agent_4': -28978.95449875767}
